Definición del Problema:
Evaluar cómo la volatilidad en el precio de activos financieros (Bitcoin) impacta el volumen de depósitos y retiros de los usuarios en una plataforma de inversión.

Objetivo: Limpiar y estructurar datos de mercado y de usuarios para su posterior visualización.

Datos necesarios: Histórico de precios (vía API) y registros de transacciones de usuarios.

Métrica de éxito: Obtener un DataFrame consolidado sin valores atípicos de sistema, listo para el análisis exploratorio (EDA).

In [1]:
import pandas as pd
import numpy as np
import requests

# Consumo de API pública de CoinGecko para histórico de BTC (90 días)
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=90"
response = requests.get(url)
raw_data = response.json()

# Parseo de la respuesta a DataFrame
precios = raw_data['prices']
df_market = pd.DataFrame(precios, columns=['timestamp', 'precio_usd'])

# Convertir timestamp a fecha legible
df_market['fecha'] = pd.to_datetime(df_market['timestamp'], unit='ms').dt.date
df_market = df_market.drop(columns=['timestamp'])

In [2]:
# Introducimos errores intencionales para demostrar la limpieza
df_market.loc[10:15, 'precio_usd'] = np.nan
df_market = pd.concat([df_market, df_market.iloc[0:5]]) 

# Limpieza real
df_market = df_market.drop_duplicates()

# Interpolación lineal para no perder los días con valores faltantes
df_market['precio_usd'] = df_market['precio_usd'].interpolate(method='linear')

# Filtramos para asegurarnos de no tener registros corruptos de precio negativo o cero
df_market = df_market[df_market['precio_usd'] > 0]

In [3]:
# Generación de datos sintéticos: simulación de transacciones de usuarios en la plataforma
np.random.seed(42)
fechas_unicas = df_market['fecha'].unique()

n_records = len(fechas_unicas)
df_usuarios = pd.DataFrame({
    'fecha': fechas_unicas,
    'nuevos_usuarios': np.random.randint(50, 500, size=n_records),
    'volumen_depositos_usd': np.random.normal(50000, 15000, size=n_records).round(2)
})

# Forzamos algunos tipos de datos para la transformación
df_usuarios['nuevos_usuarios'] = df_usuarios['nuevos_usuarios'].astype(float)
df_usuarios['nuevos_usuarios'] = df_usuarios['nuevos_usuarios'].astype(int)

# Unión (Merge) de datos de mercado con datos de usuarios
df_final = pd.merge(df_market, df_usuarios, on='fecha', how='inner')

# Derivación de columnas: variación porcentual diaria del precio
df_final['retorno_diario'] = df_final['precio_usd'].pct_change()
df_final['retorno_diario'] = df_final['retorno_diario'].fillna(0)

# Verificación final
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 2157 entries, 0 to 2156
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   precio_usd             2157 non-null   float64
 1   fecha                  2157 non-null   object 
 2   nuevos_usuarios        2157 non-null   int64  
 3   volumen_depositos_usd  2157 non-null   float64
 4   retorno_diario         2157 non-null   float64
dtypes: float64(3), int64(1), object(1)
memory usage: 84.4+ KB


In [4]:
import os

# Guardar en la estructura del proyecto
os.makedirs('data/processed', exist_ok=True)
df_final.to_csv('data/processed/df_financiero_limpio.csv', index=False)